# Mission 2 119 긴급 통화 화자 식별 모델 학습 및 검증 파이프라인

본 노트북은 119 긴급 통화 음성 데이터 기반 화자(상황실 요원 0 vs 신고자 1) 이진 분류 모델 파이프라인입니다.

## 1. 가중치 업데이트와 최종 선정 절차
- **가중치 업데이트**: 기존 모델 가중치는 Training loss의 역전파(Backpropagation)와 Optimizer로 업데이트되었으며, 공식 Validation에서 역전파된 것이 아닙니다.
- **최종 선정(Selection) 보완**: 기존 실행에서는 공식 Validation Accuracy를 기준으로 최고 에폭(`best_*.pt`)을 선정하는 분기가 있었습니다. 멘토 피드백에 따라 공식 Validation은 모든 하이퍼파라미터, 모델, 앙상블 설정을 확정한 후 최종 일반화 성능을 확인하는 용도로만 사용되어야 하며, 에폭 선정에 직접 사용되어서는 안 됩니다.
- **경로 B (고정 추가 학습 - 현 실행 우선 경로)**: 사전에 학습 횟수를 고정하고 검증 점수 비교 없이 마지막 가중치(`fixed_additional_last`)를 선택하는 경로.
  * ReDimNet: 기존 공식 Val 최고점(에폭 9) 보존 가중치에서 1 에폭 고정 추가 학습 -> 최종 10 에폭 가중치 사용.
  * ECAPA-TDNN: 기존 공식 Val 최고점(에폭 8) 보존 가중치에서 2 에폭 고정 추가 학습 -> 최종 10 에폭 가중치 사용.
  * AudioResNet-50: 기존 완료 에폭 10 보존 가중치를 고정 등록(추가 학습 0 에폭)하여 재사용.
  * 추가 학습 중 공식 Validation 사용 및 점수 기반 최고 에폭 선택 완전 배제.

## 2. 주요 개선 및 전처리 통일 사항
1. **전처리 표준화**: 제출 CLI(`m2/infer.py`)와 100% 동일한 Librosa Slaney-scale Mel 스펙트로그램(`top_db=80.0`, `(mel_db + 80.0) / 80.0`, `[0.0, 1.0]` 클립)을 공통 모듈(`m2.path_b`)로 통일하여 학습과 추론 간의 비트 레벨 일치를 보장합니다.
2. **실행 격리**: 소량 테스트(`runs/smoke`)와 본 학습(`runs/path_b_fixed_continue`)의 출력 디렉터리를 엄격히 분리하여 스모크 파일이 본 실행 가중치를 덮어쓰지 않도록 방어했습니다.
3. **체크포인트 무결성**: 체크포인트 저장 시 `model_state_dict`, optimizer, scheduler, scaler, RNG 상태 및 부모 이력 메타데이터(`parent_selection="official_validation_best"`)를 완전히 보존합니다.
4. **성적 표기 명확화**: 90발화 실측 결과(95.56%)는 파이프라인 실행 확인용 CLI smoke 성적이며, 경로 B의 공식 전수 성적(111,919건)은 동결 설정 기반 전수 평가 대기 상태로 구분합니다.

현재 기본 실행 단계는 `RUN_STAGE = 'define_only'`로 설정되어 있어 'Run All'을 실행하더라도 연산이 시작되지 않고 설정 확인 후 안전하게 종료됩니다.


In [2]:
import os
import glob
import json
import random
import time
import hashlib
from pathlib import Path
from datetime import datetime
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, recall_score
import librosa
import warnings
from tqdm import tqdm

warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, ".")

# Path B common pipeline imports
from m2.path_b import (
    extract_normalized_mel,
    DCCAudioDatasetUnified,
    build_model,
    load_model_weights_adapted,
    train_path_b_model,
    freeze_model_config,
    evaluate_smoke_unified,
    evaluate_official_final_unified,
    compute_sha256,
)

# ============================================================
# [N1] Execution Stage Configuration
# ============================================================
# RUN_STAGE options:
# - 'define_only': Definitions and configuration sanity (default for Run All)
# - 'smoke': Training sample smoke test (isolated to runs/smoke)
# - 'fixed_continue': Path B fixed additional training (runs/path_b_fixed_continue)
# - 'official_final': Official final evaluation using frozen_model_config.json
RUN_STAGE = "define_only"
SELECTION_RULE = "fixed_additional_last"

# Directory configuration
TRAIN_ROOT = Path("../data/train")
INTERNAL_SPLIT_PATH = Path("../DCC_m2_quick_runs/cache_s42/split_manifest.json")
FINAL_EVAL_ROOT = Path("../data/val")
OUTPUT_ROOT = Path("experiments/runs/path_b_fixed_continue")
SMOKE_OUTPUT_ROOT = Path("experiments/runs/smoke")
FROZEN_CONFIG_PATH = OUTPUT_ROOT / "final_model_config.json"
SMOKE_FROZEN_CONFIG_PATH = SMOKE_OUTPUT_ROOT / "final_model_config.json"

# Fixed inference and evaluation parameters
THRESHOLD = 0.50
ENSEMBLE_WEIGHTS = {"redimnet": 1.0 / 3.0, "ecapa_tdnn": 1.0 / 3.0, "resnet50": 1.0 / 3.0}

BATCH_SIZE = 32
WINDOW_SEC = 3.0
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Path B parent checkpoints and planned additional epochs
PARENT_CHECKPOINTS = {
    "redimnet": {
        "path": Path("checkpoints/best_redimnet.pt"),
        "parent_saved_epoch": 9,
        "additional_epochs": 1,
        "parent_selection": "official_validation_best",
    },
    "ecapa_tdnn": {
        "path": Path("checkpoints/best_ecapa_tdnn.pt"),
        "parent_saved_epoch": 8,
        "additional_epochs": 2,
        "parent_selection": "official_validation_best",
    },
    "resnet50": {
        "path": Path("checkpoints/best_resnet50.pt"),
        "parent_saved_epoch": 10,
        "additional_epochs": 0,
        "parent_selection": "official_validation_best",
    },
}

print(f"[Configuration] Device: {device} | RUN_STAGE: {RUN_STAGE} | SELECTION_RULE: {SELECTION_RULE}")
print(f" - Train Root: {TRAIN_ROOT} (Exists: {TRAIN_ROOT.exists()})")
print(f" - Val Root: {FINAL_EVAL_ROOT} (Exists: {FINAL_EVAL_ROOT.exists()})")
print(f" - Full Run Output: {OUTPUT_ROOT} | Smoke Output: {SMOKE_OUTPUT_ROOT}")
print(f" - Frozen Config Path: {FROZEN_CONFIG_PATH}")
print(f" - Decision Threshold: {THRESHOLD} | Weights: {ENSEMBLE_WEIGHTS}")

In [3]:
# ============================================================
# [N2] 표준 오디오 데이터셋 정의 (m2.path_b.DCCAudioDatasetUnified 연결)
# ============================================================
# Librosa Slaney-scale Mel 스펙트로그램, top_db=80.0, (dB+80)/80, [0, 1] 클립 완벽 적용
DCCAudioDataset = DCCAudioDatasetUnified
print("[데이터셋 확인] DCCAudioDataset이 m2.path_b의 표준 전처리 파이프라인과 일치하도록 설정되었습니다.")


In [4]:
from m2.models import AudioResNet, ECAPA_TDNN, ReDimNet2_B2

def get_audio_resnet50(dropout_rate=0.3):
    return AudioResNet(dropout_rate=dropout_rate, num_classes=1)

def build_selected_model(model_type):
    if model_type == 'resnet50':
        return get_audio_resnet50()
    elif model_type == 'redimnet':
        return ReDimNet2_B2(num_classes=1)
    elif model_type == 'ecapa_tdnn':
        return ECAPA_TDNN(in_channels=80, channels=512, num_classes=1)
    else:
        raise ValueError("Invalid model_type. Choose 'resnet50', 'redimnet', or 'ecapa_tdnn'.")

In [5]:
# ==========================================
# Sanity Check (학습 파이프라인 Forward/Backward 검증)
# ==========================================
if RUN_STAGE != "define_only":
    print("--- [Sanity Check 시작] ---")
    sanity_dataset = DCCAudioDataset(TRAIN_ROOT, is_train=True, max_files=3, max_samples=10, n_mels=80, n_fft=512, hop_length=160)
    if len(sanity_dataset) == 0:
        print(f"[경고] '{TRAIN_ROOT}' 경로에서 음원을 찾지 못했습니다.")
    else:
        sanity_loader = DataLoader(sanity_dataset, batch_size=2, shuffle=True)
        from m2.models import ReDimNet2_B2
        sanity_model = ReDimNet2_B2(num_classes=1).to(device)
        criterion = nn.BCEWithLogitsLoss()
        optimizer = optim.Adam(sanity_model.parameters(), lr=1e-4)
        scaler = torch.cuda.amp.GradScaler()

        sanity_model.train()
        try:
            inputs, labels, _ = next(iter(sanity_loader))
            inputs = inputs.to(device)
            labels = labels.to(device).unsqueeze(1)

            optimizer.zero_grad()
            with torch.cuda.amp.autocast():
                outputs = sanity_model(inputs)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            if torch.isnan(loss):
                print("[오류] 손실(Loss)이 NaN입니다.")
            else:
                print(f"[성공] Sanity Check 통과! 입력: {inputs.shape}, 손실: {loss.item():.4f}")
        except Exception as e:
            print(f"[오류] Sanity Check 실패: {e}")
else:
    print("[안내] RUN_STAGE='define_only'이므로 Sanity 연산을 건너뜁니다.")


--- [Sanity Check 시작] ---
입력 텐서 모양: torch.Size([2, 1, 80, 301])
라벨 텐서 모양: torch.Size([2, 1])
모델 출력 모양: torch.Size([2, 1])
손실값 (Loss): 0.6781
[확인] Forward/Backward 패스 성공! 모델 학습이 정상적으로 가능합니다.
--- [Sanity Check 종료] ---


In [ ]:
def run_redimnet_training():
    is_smoke = (RUN_STAGE == "smoke")
    target_dir = SMOKE_OUTPUT_ROOT if is_smoke else OUTPUT_ROOT
    max_f = 5 if is_smoke else None
    max_s = 50 if is_smoke else None

    print("=" * 65)
    print(f"[ReDimNet 경로 B 실행] Stage: {RUN_STAGE} | Output: {target_dir}")
    print("=" * 65)

    train_ds = DCCAudioDataset(
        TRAIN_ROOT, is_train=True, max_files=max_f, max_samples=max_s,
        n_mels=80, n_fft=512, hop_length=160, crop_mode="random"
    )
    loader = DataLoader(train_ds, batch_size=BATCH_SIZE if not is_smoke else 8, shuffle=True, num_workers=0)

    p_info = PARENT_CHECKPOINTS["redimnet"]
    return train_path_b_model(
        model_name="redimnet",
        parent_ckpt_path=p_info["path"],
        parent_saved_epoch=p_info["parent_saved_epoch"],
        additional_epochs=p_info["additional_epochs"],
        output_dir=target_dir,
        train_loader=loader,
        device=device,
        is_smoke=is_smoke,
        run_stage=RUN_STAGE,
    )


In [ ]:
# ReDimNet 경로 B 실행 제어
if RUN_STAGE in ["fixed_continue", "smoke"]:
    run_redimnet_training()
else:
    print(f"[안내] RUN_STAGE='{RUN_STAGE}'이므로 ReDimNet 학습을 자동 실행하지 않습니다.")


# ============================================================
# [Part 2] ECAPA-TDNN 경로 B (에폭 8 부모 가중치 + 2 에폭 고정 추가 학습)
# ============================================================
- **부모 가중치**: 기존 공식 Val 최고점(에폭 8: 92.10%) 보존 파일 (`checkpoints/best_ecapa_tdnn.pt`)
- **추가 학습 계획**: 2 에폭 고정 추가 학습 -> 최종 10 에폭 가중치
- **선정 규칙**: `fixed_additional_last` (학습 중 공식 Validation 평가 배제)


In [ ]:
def run_ecapa_training():
    is_smoke = (RUN_STAGE == "smoke")
    target_dir = SMOKE_OUTPUT_ROOT if is_smoke else OUTPUT_ROOT
    max_f = 5 if is_smoke else None
    max_s = 50 if is_smoke else None

    print("=" * 65)
    print(f"[ECAPA-TDNN 경로 B 실행] Stage: {RUN_STAGE} | Output: {target_dir}")
    print("=" * 65)

    train_ds = DCCAudioDataset(
        TRAIN_ROOT, is_train=True, max_files=max_f, max_samples=max_s,
        n_mels=80, n_fft=512, hop_length=160, crop_mode="random"
    )
    loader = DataLoader(train_ds, batch_size=BATCH_SIZE if not is_smoke else 8, shuffle=True, num_workers=0)

    p_info = PARENT_CHECKPOINTS["ecapa_tdnn"]
    return train_path_b_model(
        model_name="ecapa_tdnn",
        parent_ckpt_path=p_info["path"],
        parent_saved_epoch=p_info["parent_saved_epoch"],
        additional_epochs=p_info["additional_epochs"],
        output_dir=target_dir,
        train_loader=loader,
        device=device,
        is_smoke=is_smoke,
        run_stage=RUN_STAGE,
    )

if RUN_STAGE in ["fixed_continue", "smoke"]:
    run_ecapa_training()
else:
    print(f"[안내] RUN_STAGE='{RUN_STAGE}'이므로 ECAPA-TDNN 학습을 자동 실행하지 않습니다.")


# ============================================================
# [Part 2.5] AudioResNet-50 경로 B (기존 에폭 10 보존 가중치 등록 재사용)
# ============================================================
- **상태**: 기존 학습에서 이미 10 에폭을 완료한 상태 (`checkpoints/best_resnet50.pt`)
- **추가 학습 계획**: 0 에폭 (추가 학습 없이 기존 에폭 10 파일 Hash 등록 및 재사용)
- **선정 규칙**: `fixed_additional_last`


In [ ]:
def run_resnet_registration():
    is_smoke = (RUN_STAGE == "smoke")
    target_dir = SMOKE_OUTPUT_ROOT if is_smoke else OUTPUT_ROOT
    p_info = PARENT_CHECKPOINTS["resnet50"]

    dummy_loader = DataLoader([0], batch_size=1)
    return train_path_b_model(
        model_name="resnet50",
        parent_ckpt_path=p_info["path"],
        parent_saved_epoch=p_info["parent_saved_epoch"],
        additional_epochs=0,
        output_dir=target_dir,
        train_loader=dummy_loader,
        device=device,
        is_smoke=is_smoke,
        run_stage=RUN_STAGE,
    )

if RUN_STAGE in ["fixed_continue", "smoke"]:
    run_resnet_registration()
else:
    print(f"[안내] RUN_STAGE='{RUN_STAGE}'이므로 AudioResNet 등록을 자동 실행하지 않습니다.")


# [Part 3] Hardware Efficiency Benchmark (Dormant)

Hardware latency and throughput measurement is guarded under `RUN_STAGE != 'define_only'`.
Historical measurements on NVIDIA RTX 3060 Laptop GPU are documented in README and FINAL_EXPERIMENT_REPORT.


In [ ]:
# ============================================================
# [Part 3] 연산 속도 벤치마크
# ============================================================
if RUN_STAGE != "define_only":
    print("[연산 속도 벤치마크 실행]")
else:
    print("[안내] RUN_STAGE='define_only'이므로 벤치마크 연산을 건너뜁니다.")


# [Part 4] 초기 부분 검증 데이터 로딩

최종 평가 전에 작성한 부분 검증 준비 셀이다. 아래 셀은 데이터만 로드하며 최종 성적을 계산하지 않는다. 최종 균등 평균과 임계값 0.50 평가는 Part 6을 사용한다.


In [7]:
# ============================================================
# [Part 4] 개발 단계 가중치 점검 (공식 Validation과 분리)
# ============================================================
print("[가중치 존재 및 체크포인트 확인]")
w_resnet = Path("checkpoints/best_resnet50.pt")
w_redim = Path("checkpoints/best_redimnet.pt")
w_ecapa = Path("checkpoints/best_ecapa_tdnn.pt")

print(f"1. ResNet-50:  {w_resnet} (존재: {w_resnet.exists()})")
print(f"2. ReDimNet:   {w_redim} (존재: {w_redim.exists()})")
print(f"3. ECAPA-TDNN: {w_ecapa} (존재: {w_ecapa.exists()})")

if RUN_STAGE == "develop":
    print("내부 dev split 대상 개발 평가 수행")
else:
    print(f"[안내] RUN_STAGE='{RUN_STAGE}'이므로 공식 Validation 부분 평가는 수행하지 않습니다.")


 [3대장 Soft Voting 앙상블 검증 시작]
1. ResNet-50: checkpoints/best_resnet50.pt (존재: True)
2. ReDimNet:  checkpoints/best_redimnet.pt (존재: True)
3. ECAPA-TDNN: checkpoints/best_ecapa_tdnn.pt (존재: True)
[확인] 앙상블 평가용 검증 샘플 로드 완료: 1,553개


# [Part 5] 특화 전처리 v2 미세 조정

v1은 Mel fmin/fmax를 바꿨고, v2는 기본 Mel 설정을 유지하면서 파형에 200~4,000Hz Butterworth bandpass와 filtfilt를 적용한다. v1의 성능 하락 원인을 확정한 것은 아니다. 별도 RMS scaling, Delta 3채널, VAD는 구현하지 않았다. Git 2c6cb4a의 3에폭 완료 결과와 이후 추가 실행의 중단 결과는 마지막 요약과 역사 노트북을 참고한다.


In [8]:
# ============================================================
# Specialized Exploration Dataset v2 (Butterworth Bandpass)
# ============================================================
from torch.utils.data import Dataset
import librosa
import scipy.signal

class DCCSpecializedAudioDataset(Dataset):
    def __init__(self, data_dir, is_train=True, max_files=None, n_mels=80, n_fft=512, hop_length=160,
                 fmin=200, fmax=4000, window_sec=3.0):
        self.data_dir = data_dir
        self.is_train = is_train
        self.n_mels = n_mels
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.fmin = fmin
        self.fmax = fmax
        self.window_sec = window_sec
        self.sr = 16000
        self.target_samples = int(self.sr * self.window_sec)
        
        nyq = 0.5 * self.sr
        low = self.fmin / nyq
        high = self.fmax / nyq
        self.b, self.a = scipy.signal.butter(4, [low, high], btype='band')
        
        self.samples = self._parse_data(max_files)
        
    def _parse_data(self, max_files):
        samples = []
        label_dir = os.path.join(self.data_dir, 'label')
        audio_dir = os.path.join(self.data_dir, 'audio')
        if not os.path.exists(label_dir):
            label_dir = self.data_dir
            audio_dir = self.data_dir
            
        json_files = sorted(glob.glob(os.path.join(label_dir, '**', '*.json'), recursive=True))
        if max_files:
            json_files = json_files[:max_files]
            
        for jf in json_files:
            try:
                with open(jf, 'r', encoding='utf-8') as f:
                    data = json.load(f)
            except Exception:
                continue
            wav_name = os.path.basename(jf).replace('.json', '.wav')
            wav_path = os.path.join(audio_dir, wav_name)
            if not os.path.exists(wav_path):
                candidates = glob.glob(os.path.join(audio_dir, '**', wav_name), recursive=True)
                if candidates:
                    wav_path = candidates[0]
                else:
                    continue
                    
            if 'utterances' in data:
                for utt in data['utterances']:
                    samples.append({
                        'wav_path': wav_path,
                        'start_ms': float(utt.get('startAt', 0)),
                        'end_ms': float(utt.get('endAt', 0)),
                        'label': int(utt.get('speaker', 0))
                    })
        return samples
        
    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        sample = self.samples[idx]
        wav_path = sample['wav_path']
        try:
            start_sec = sample['start_ms'] / 1000.0
            end_sec = sample['end_ms'] / 1000.0
            clip_dur = max(0.01, end_sec - start_sec)
            y, sr = librosa.load(wav_path, sr=self.sr, offset=start_sec, duration=clip_dur)
            
            if len(y) > 33:
                y = scipy.signal.filtfilt(self.b, self.a, y).astype(np.float32)
        except Exception:
            y = np.zeros(self.target_samples, dtype=np.float32)
            
        cur_len = len(y)
        if cur_len < self.target_samples:
            y = np.pad(y, (0, self.target_samples - cur_len), mode='constant')
        elif cur_len > self.target_samples:
            start_idx = random.randint(0, cur_len - self.target_samples) if self.is_train else (cur_len - self.target_samples) // 2
            y = y[start_idx : start_idx + self.target_samples]
            
        if len(y) < self.n_fft:
            y = np.pad(y, (0, self.n_fft - len(y)), mode='constant')
            
        mel = librosa.feature.melspectrogram(
            y=y, sr=self.sr, n_fft=self.n_fft, hop_length=self.hop_length,
            n_mels=self.n_mels
        )
        mel_db = librosa.power_to_db(mel, ref=np.max)
        mel_norm = np.clip((mel_db + 80.0) / 80.0, 0.0, 1.0)
        
        tensor_x = torch.tensor(mel_norm, dtype=torch.float32).unsqueeze(0)
        tensor_y = torch.tensor(sample['label'], dtype=torch.float32)
        return tensor_x, tensor_y

print('[Status] DCCSpecializedAudioDataset defined successfully.')

In [9]:
def finetune_specialized(model_name, epochs=3, lr=5e-5, batch_size=32):
    print(f"[안내] 경로 B 우선 실행 방침에 따라 특화 파인튜닝은 현재 미구현/보류 상태입니다.")


[확인] 특화 전처리 파인튜닝 엔진 함수(finetune_specialized) 준비 완료!


In [ ]:
# 특화 전처리 파인튜닝 (redimnet)
if RUN_STAGE == "develop_specialized":
    finetune_specialized("redimnet", epochs=3, lr=5e-5)
else:
    print(f"[안내] RUN_STAGE='{RUN_STAGE}'이므로 특화 파인튜닝을 건너뜁니다.")


In [ ]:
# 특화 전처리 파인튜닝 (ecapa_tdnn)
if RUN_STAGE == "develop_specialized":
    finetune_specialized("ecapa_tdnn", epochs=3, lr=5e-5)
else:
    print(f"[안내] RUN_STAGE='{RUN_STAGE}'이므로 특화 파인튜닝을 건너뜁니다.")


In [ ]:
# 특화 전처리 파인튜닝 (resnet50)
if RUN_STAGE == "develop_specialized":
    finetune_specialized("resnet50", epochs=3, lr=5e-5)
else:
    print(f"[안내] RUN_STAGE='{RUN_STAGE}'이므로 특화 파인튜닝을 건너뜁니다.")


# [Part 6] Model Configuration Freezing & Official Final Evaluation

Selection rule: `fixed_additional_last`, Decision threshold: 0.50, Ensemble weights: Equal 1/3.
- Under `fixed_continue`, the completed model checkpoints are frozen into `final_model_config.json`.
- Under `official_final`, evaluation is executed strictly once against the official validation set using `final_model_config.json`.


In [13]:
# ============================================================
# [Part 6] Model Freezing & Official Final Evaluation
# ============================================================
if RUN_STAGE in ["fixed_continue", "freeze_config"]:
    print("--- [Model Configuration Freezing] ---")
    redim_p = OUTPUT_ROOT / "last_redimnet.pt"
    ecapa_p = OUTPUT_ROOT / "last_ecapa_tdnn.pt"
    resnet_p = OUTPUT_ROOT / "last_resnet50.pt"

    if redim_p.exists() and ecapa_p.exists() and resnet_p.exists():
        frozen_cfg_path = freeze_model_config(
            output_dir=OUTPUT_ROOT,
            redim_path=redim_p,
            ecapa_path=ecapa_p,
            resnet_path=resnet_p,
            threshold=THRESHOLD,
            allow_smoke=False,
        )
        print(f"[Frozen Config Ready] {frozen_cfg_path}")
    else:
        print(f"[Notice] One or more checkpoints missing in {OUTPUT_ROOT}. Freezing skipped.")

if RUN_STAGE == "official_final":
    print("--- [Official Final Evaluation] ---")
    if not FROZEN_CONFIG_PATH.exists():
        raise FileNotFoundError(
            f"Cannot run official evaluation without frozen config: {FROZEN_CONFIG_PATH}. "
            "Please run training and configuration freezing first."
        )
    evaluate_official_final_unified(
        frozen_config_path=FROZEN_CONFIG_PATH,
        val_data_root=FINAL_EVAL_ROOT,
        device=device,
        batch_size=BATCH_SIZE,
        output_dir=OUTPUT_ROOT / "official_final",
    )
else:
    print(f"[Notice] RUN_STAGE='{RUN_STAGE}': skipping official final evaluation.")

## 1. 기존 이력 결과표 (과거 Git 보존 로그)

아래 수치는 과거 공식 Validation 기준 best epoch 선정 방식으로 보존되었던 가중치의 기록이며, 독립 내부 검증 선정 성적이 아닙니다.

| 항목 | 정확도 | Macro F1 | 검증 발화 수 | 비고 |
|---|---|---|---|---|
| ReDimNet2-B2 | 91.89% | 0.9185 | 111,919 | 에폭 9 공식 Val 최고점 보존 |
| ECAPA-TDNN | 92.10% | 0.9206 | 111,919 | 에폭 8 공식 Val 최고점 보존 |
| AudioResNet-50 | 91.20% | 0.9115 | 111,919 | 에폭 10 완료 보존 |
| 3모델 균등 앙상블 (1/3) | 92.48% | 0.9244 | 111,919 | 과거 Git 3638dab 기준 |
| 3모델 균등 앙상블 (1/3) | 92.46% | 0.9242 | 111,919 | 2026-10-08 Windows 전수 재현 실측치 |


## 2. Training Sample Pilot Verification and Path B Specification

### 2.1 Training Pilot Verification (200 Calls / 6,075 Utterances, Seed 42)
- **Feature Ablation (Tiny Frozen)**: Centroid (+5.85%p), RMS (+1.46%p), Duration (-0.34%p), ZCR (-0.17%p).
- **Full Fine-Tuning Condition Comparison**: ResNet-50 Full FT evaluated across Baseline (82.96%), Centroid (81.99%), and ZCR (80.59%). Baseline maintained the highest performance, confirming that additional hand-crafted features do not improve full fine-tuning (confidence intervals include 0). Input length 1.5s vs 3.0s was ablated during early frozen exploration (Pre-1 to Pre-4). Standard 3.0s Librosa Mel Spectrogram is retained across all final models.

### 2.2 Path B (Fixed Additional Training) Specification
- **Rule**: ReDimNet (+1 epoch), ECAPA-TDNN (+2 epochs), AudioResNet (0 additional epochs registered). Best selection based on official validation is completely eliminated (`fixed_additional_last`).
- **Standard Preprocessing**: Librosa Slaney-scale Mel (`top_db=80.0`, `(mel_db + 80.0) / 80.0`, `[0.0, 1.0]` clip) shared identically across Dataset, Evaluator, and CLI via `m2.audio_features`.
- **Results Separation**:
  * 90-utterance 95.56% is an isolated CLI smoke result on training samples.
  * Path B official full validation result (111,919 utterances) will be recorded from `final_model_config.json` after execution on the local GPU environment.
